# LeetCode #99: Recover Binary Search Tree

https://leetcode.com/problems/recover-binary-search-tree/

## Comparison of Approaches

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Collect inorder, sort, overwrite | O(n log n) | O(n) | Get inorder values, sort, write back |
| **DFS Inorder (find violations) ★** | **O(n)** | **O(n)** | Track prev; first violation sets `first`, every violation updates `second` |
| Morris Traversal | O(n) | O(1) | Same logic; threaded links for O(1) space |

---

## Understanding the Methods

### Brute Force (Sort)
Collect inorder values, sort them, overwrite node values in inorder order. O(n log n).

### Optimal: DFS Inorder ★
BST inorder traversal should be strictly sorted. A single swap creates either:
- 1 violation (adjacent nodes swapped): [1, 3, 2, 4] → `first=3, second=2`
- 2 violations (non-adjacent): [1, 6, 3, 4, 5, 2] → `first=6` (1st violation's prev), `second=2` (last violation's curr)

Algorithm: track `prev` node during inorder. When prev.val > curr.val: set first=prev (only first time), always set second=curr. After traversal, swap first.val and second.val.

**Constraints:**
* Number of nodes: [2, 1000]
* Node values: [-2^31, 2^31 - 1]
* Exactly two nodes are swapped

## Solutions

### C#

In [ ]:
public class Solution {
    TreeNode first = null, second = null, prev = null;

    public void RecoverTree(TreeNode root) {
        Inorder(root);
        (first.val, second.val) = (second.val, first.val);
    }

    void Inorder(TreeNode node) {
        if (node == null) return;
        Inorder(node.left);
        if (prev != null && prev.val > node.val) {
            if (first == null) first = prev; // first violation: misplaced bigger node
            second = node;                    // keep updating: misplaced smaller node
        }
        prev = node;
        Inorder(node.right);
    }
}

### Python

In [ ]:
class Solution:
    def recoverTree(self, root) -> None:
        self.first = self.second = self.prev = None

        def inorder(node):
            if not node:
                return
            inorder(node.left)
            if self.prev and self.prev.val > node.val:
                if not self.first:
                    self.first = self.prev
                self.second = node
            self.prev = node
            inorder(node.right)

        inorder(root)
        self.first.val, self.second.val = self.second.val, self.first.val

### Go

In [ ]:
var first, second, prev *TreeNode

func recoverTree(root *TreeNode) {
    first, second, prev = nil, nil, nil
    inorder(root)
    first.Val, second.Val = second.Val, first.Val
}

func inorder(node *TreeNode) {
    if node == nil {
        return
    }
    inorder(node.Left)
    if prev != nil && prev.Val > node.Val {
        if first == nil {
            first = prev
        }
        second = node
    }
    prev = node
    inorder(node.Right)
}

### Rust

In [ ]:
impl Solution {
    pub fn recover_tree(root: &mut Option<Rc<RefCell<TreeNode>>>) {
        let mut first: Option<Rc<RefCell<TreeNode>>> = None;
        let mut second: Option<Rc<RefCell<TreeNode>>> = None;
        let mut prev: Option<Rc<RefCell<TreeNode>>> = None;

        // Collect inorder values and node refs
        let mut nodes: Vec<Rc<RefCell<TreeNode>>> = Vec::new();
        Self::collect(root, &mut nodes);

        for i in 1..nodes.len() {
            let a = nodes[i-1].borrow().val;
            let b = nodes[i].borrow().val;
            if a > b {
                if first.is_none() { first = Some(nodes[i-1].clone()); }
                second = Some(nodes[i].clone());
            }
        }

        if let (Some(f), Some(s)) = (first, second) {
            let fv = f.borrow().val;
            let sv = s.borrow().val;
            f.borrow_mut().val = sv;
            s.borrow_mut().val = fv;
        }
    }

    fn collect(node: &Option<Rc<RefCell<TreeNode>>>, nodes: &mut Vec<Rc<RefCell<TreeNode>>>) {
        if let Some(n) = node {
            Self::collect(&n.borrow().left.clone(), nodes);
            nodes.push(n.clone());
            Self::collect(&n.borrow().right.clone(), nodes);
        }
    }
}

## Example Scenarios

### 1. Common Case — Non-Adjacent Swap
**Input:** `[3,1,4,null,null,2]` (3 and 2 swapped)
Inorder: [1,3,2,4]. Violation: 3>2 → first=3, second=2. Swap → [1,2,3,4].
**Output:** Corrected BST [2,1,4,null,null,3]

### 2. Slightly Complex — Adjacent Swap
**Input:** `[1,3,null,null,2]` (3 and 2 swapped in adjacent positions)
Inorder: [3,2,1]. One violation: 3>2 → first=3, second=2. Swap.
**Output:** Corrected BST [1,2,null,null,3]

### 3. Edge Case — Root Swapped with Rightmost
**Input:** Root has largest value; rightmost leaf has root's correct value.
Two inorder violations; both identified correctly.

### 4. Edge Case — Two-Node Tree
**Input:** `[2,1]` (already valid BST, no swap needed)
No violation detected. But the problem guarantees exactly two nodes are swapped.

### 5. Almost-Impossible but Plausible — Swap Creates Multiple Violations
Non-adjacent swap always creates exactly 2 violations. Adjacent swap creates 1. The algorithm handles both by always updating `second`.


*Infographic will be added in a future update.*